# Building a Prettier Chart with Bokeh

## The standard Numpy and Pandas imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Additional Imports to demo using Bokeh

In [2]:
from math import pi

from bokeh.palettes import Category20c
from bokeh.plotting import figure, output_notebook, show
from bokeh.io import reset_output
from bokeh.transform import cumsum
from IPython.display import clear_output

output_notebook(hide_banner=True)

## Load the data

Data is from the Origins Game Fair
* main site - https://www.originsgamefair.com/
* schedule - https://tabletop.events/conventions/origins-game-fair-2026/schedule
* scheule CSV - https://s3.amazonaws.com/conventionshare.tabletop.events/schedules/origins-game-fair-2026-schedule.csv

In [3]:
# read_csv has numerous options that let you dial in how 
# the data is loaded i.e. loading only certain columns, 
# specifying data types, specifying date/time format
df = pd.read_csv('origins-game-fair-2026-schedule.csv')

## Look at the values in 1 column

In [4]:
df['Event Type'].value_counts()[:5]

Event Type
Board Game          3321
Roleplaying Game    2447
Miniatures Game      795
Card Game            594
Escape Games         343
Name: count, dtype: int64

## Create a copy of the series

In [5]:
s = df['Event Type'].value_counts().copy(deep=True)
s[:5]

Event Type
Board Game          3321
Roleplaying Game    2447
Miniatures Game      795
Card Game            594
Escape Games         343
Name: count, dtype: int64

## Examine the Index of the Series

In [6]:
s.index

Index(['Board Game', 'Roleplaying Game', 'Miniatures Game', 'Card Game',
       'Escape Games', 'Workshop', 'Social Deduction/Deception',
       'Live Action Roleplaying Game', 'Seminars and Panels', 'Eclectic',
       'Entertainment', 'Classic Games',
       'Game-Based Education & Therapy Conference', 'Puzzle Games',
       'GAMA Trade Day'],
      dtype='str', name='Event Type')

## Create a DataFrame from the series (table of 1 column)

In [8]:
df2 = pd.DataFrame({'Event Type': s})
df2.head(3)

,Event Type
Event Type,
Board Game,3321
Roleplaying Game,2447
Miniatures Game,795


## The DataFrame uses the same Index as the Series

In [9]:
df2.index[:5]

Index(['Board Game', 'Roleplaying Game', 'Miniatures Game', 'Card Game',
       'Escape Games'],
      dtype='str', name='Event Type')

## Index and column have same name, so we rename the column

In [12]:
df2 = df2.rename(columns={'Event Type': 'count'})
df2.head(3)

,count,angle,color,prct
Event Type,,,,
Board Game,3321,2.407298,#3182bd,38.313336
Roleplaying Game,2447,1.773760,#6baed6,28.230272
Miniatures Game,795,0.576273,#9ecae1,9.171666


## Compute angle, color, and percent

In [13]:
total = df2['count'].sum()
df2['angle'] = df2['count'] / total * 2 * pi
df2['color'] = Category20c[len(df2)]
df2['prct'] = df2['count'] / total * 100
df2.head(3)

,count,angle,color,prct
Event Type,,,,
Board Game,3321,2.407298,#3182bd,38.313336
Roleplaying Game,2447,1.773760,#6baed6,28.230272
Miniatures Game,795,0.576273,#9ecae1,9.171666


In [14]:
p = figure(
    height=350, width=700, 
    title="Pie Chart",
    toolbar_location=None,
    tools="hover",
    tooltips="@{Event Type}: @count (@{prct}%)",
    x_range=(-0.5, 1.0),
)

p.wedge(x=0, y=1, radius=0.2,
        start_angle=cumsum('angle', include_zero=True), end_angle=cumsum('angle'),
        line_color="white", fill_color='color', legend_field='Event Type', source=df2)

p.axis.axis_label = None
p.axis.visible = False
p.grid.grid_line_color = None

show(p)